## Setup

Requires `OPENAI_API_KEY` in a `.env` file (and `OPENAI_BASE_URL` if you use a proxy such as Vocareum).
The databases (`data/external/cultpass.db` and `data/core/udahub.db`) are included, so you can run this notebook straight away.
Re-run `01_external_db_setup.ipynb` and `02_core_db_setup.ipynb` only to reset the data: the scenarios below make real bookings and cancellations and store tickets and memories.

In [ ]:
import logging
from dotenv import load_dotenv
from utils import chat_interface

In [ ]:
load_dotenv()

# Each scenario's decision trace is printed by `show()` below. To also stream the agents' decisions
# live (useful in the interactive chat), uncomment:
# logging.basicConfig(format="%(message)s")
# logging.getLogger("udahub").setLevel(logging.INFO)

## Run

In [ ]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

from agentic.workflow import orchestrator

**Architecture** (see `agentic/design/README.md`)

In [ ]:
print(orchestrator.get_graph().draw_mermaid())

## Scenarios

Each scenario uses its own ticket id (= LangGraph `thread_id`). `process_ticket` sends a message plus optional ticket metadata;
`show` prints the decision trace and the reply.

In [ ]:
from agentic.workflow import process_ticket


def show(state):
    for step in state["trace"]:
        print("  ", step)
    print("\nAssistant:", state["final_response"], "\n")

**1. Knowledge question from an unknown customer → RAG resolver**

In [ ]:
show(process_ticket("Hi! What's included in a CultPass subscription? Are premium events extra?", "demo-1"))

**2. Account action for a known customer → account agent with tools**

Cathy (basic tier) asks about her reservations, then cancels one in the same session (short-term memory).

In [ ]:
cathy = {"external_user_id": "88382b", "channel": "chat"}
show(process_ticket("Which reservations do I have coming up?", "demo-2", cathy))

In [ ]:
show(process_ticket("Please cancel that one, I can't make it.", "demo-2"))

**3. Guest needs an account action → asks for identification, then continues on the same thread**

In [ ]:
show(process_ticket("I'd like to book the Samba Night at Lapa.", "demo-3"))

In [ ]:
show(process_ticket("Sure, it's bob.stone@granite.com", "demo-3"))

**4. Blocked account → escalated to Trust & Safety without revealing the reason**

In [ ]:
show(process_ticket("I can't log into my account, it keeps failing!", "demo-4", {"email": "alice.kingsley@wonderland.com"}))

**5. Refund request → registered via the internal refund tool and escalated for approval (policy)**

In [ ]:
show(process_ticket("I was charged twice for my subscription this month. I want my money back.", "demo-5",
                    {"external_user_id": "e6376d", "channel": "email", "urgency": "high"}))

**6. Long-term memory: Bob had a login ticket 3 days ago, and we know his preferences**

The agents recall his previous ticket (password reset already tried) and his stored preferences.

In [ ]:
bob = {"email": "bob.stone@granite.com"}
show(process_ticket("I still can't log in, the password reset email never arrives.", "demo-6", bob))

In [ ]:
show(process_ticket("Can you recommend an experience for me to book this week?", "demo-7", bob))

In [ ]:
# Long-term memories stored for Bob across tickets
from agentic.tools import ticket_tools
state = orchestrator.get_state({"configurable": {"thread_id": "demo-7"}}).values
ticket_tools.get_memories(state["customer"]["udahub_user_id"])

In [ ]:
# Explicit request for a human → immediate escalation
show(process_ticket("This is ridiculous, I want to talk to a real person now.", "demo-8", {"external_user_id": "88382b"}))

## Interactive chat

Type `quit` to leave. The thread id is the ticket id.

In [ ]:
chat_interface(orchestrator, "1")

In [ ]:
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "1",
        }
    }
))[0].values["messages"]